# AI Layer: Customer Recommendation and Executive Summary

This notebook builds an end-to-end AI layer for the customer intelligence workflow. It loads the processed outputs from the earlier analytical modules, validates them, creates a ranked recommendation table, generates an executive summary, and saves the outputs for downstream use.

## What this notebook does
- Loads the processed customer, churn, CLV, persona, and revenue-at-risk datasets
- Validates data quality and merges the signals into a single customer view
- Uses Groq when a valid API key is available, with a deterministic fallback when it is not
- Creates customer recommendations and an executive summary
- Saves CSV, Markdown, and chart outputs in the processed data folder

In [1]:
import os
import json
import logging
import warnings
from pathlib import Path
from typing import Optional, Dict, Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

warnings.filterwarnings("ignore")

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("ai_layer")

# Resolve the project root robustly so the notebook works from either the repo root or the notebooks folder
project_root = Path.cwd()
for candidate in [project_root, project_root.parent]:
    if (candidate / "data" / "processed").exists():
        project_root = candidate
        break

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Processed data directory: {processed_dir}")

Project root: c:\Users\vkt58\OneDrive\Desktop\Customer-Intelligence-Revenue-Growth
Processed data directory: c:\Users\vkt58\OneDrive\Desktop\Customer-Intelligence-Revenue-Growth\data\processed


In [2]:
required_files = {
    "churn": processed_dir / "customer_churn_predictions.csv",
    "clv": processed_dir / "customer_clv_predictions.csv",
    "personas": processed_dir / "customer_personas.csv",
    "risk": processed_dir / "revenue_at_risk_analysis.csv",
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

churn_df = pd.read_csv(required_files["churn"])
clv_df = pd.read_csv(required_files["clv"])
personas_df = pd.read_csv(required_files["personas"])
risk_df = pd.read_csv(required_files["risk"])

print("Loaded datasets:")
for name, df in [("churn", churn_df), ("clv", clv_df), ("personas", personas_df), ("risk", risk_df)]:
    print(f"- {name}: {df.shape[0]} rows, {df.shape[1]} columns")

# Basic validation
for name, df in [("churn", churn_df), ("clv", clv_df), ("personas", personas_df), ("risk", risk_df)]:
    missing = df.isna().sum().sum()
    print(f"{name} missing values: {missing}")

# Merge the main business signals into a single customer view
persona_col = "Customer_Persona" if "Customer_Persona" in personas_df.columns else "Persona_ID"
clv_col = "Predicted_CLV" if "Predicted_CLV" in clv_df.columns else "Future_CLV"

merged_df = churn_df.rename(columns={"Customer_Persona": "Customer_Persona_churn"})
merged_df = merged_df.merge(clv_df[["CustomerID", clv_col]], on="CustomerID", how="left")
merged_df = merged_df.merge(
    personas_df[["CustomerID", persona_col]].rename(columns={persona_col: "Customer_Persona_personas"}),
    on="CustomerID",
    how="left",
)
merged_df = merged_df.merge(
    risk_df[["CustomerID", "Revenue_at_Risk", "Risk_Score", "Risk_Category", "Customer_Action"]],
    on="CustomerID",
    how="left",
)

merged_df["Customer_Persona"] = merged_df["Customer_Persona_personas"].fillna(merged_df["Customer_Persona_churn"])
merged_df = merged_df.drop(columns=["Customer_Persona_churn", "Customer_Persona_personas"])
merged_df = merged_df.rename(columns={clv_col: "Predicted_CLV"})

# Create a composite priority score for ranking
merged_df["CLV_Score"] = (merged_df["Predicted_CLV"] / merged_df["Predicted_CLV"].max()).fillna(0)
merged_df["Churn_Score"] = merged_df["Churn_Probability"].fillna(0)
merged_df["Priority_Score"] = (
    0.45 * merged_df["Risk_Score"].fillna(0)
    + 0.35 * merged_df["CLV_Score"].fillna(0)
    + 0.20 * merged_df["Churn_Score"].fillna(0)
)

merged_df = merged_df.sort_values("Priority_Score", ascending=False).reset_index(drop=True)
merged_df.head()

Loaded datasets:
- churn: 3365 rows, 14 columns
- clv: 3365 rows, 15 columns
- personas: 3365 rows, 12 columns
- risk: 3365 rows, 13 columns
churn missing values: 0
clv missing values: 0
personas missing values: 0
risk missing values: 0


,CustomerID,Recency,Frequency,Monetary,Average_Order_Value,R_Score,F_Score,M_Score,RFM_Score,Segment,...,Churn_Probability,Predicted_CLV,Revenue_at_Risk,Risk_Score,Risk_Category,Customer_Action,Customer_Persona,CLV_Score,Churn_Score,Priority_Score
0,17450.0,94,23,86301.54,3752.24,5,5,5,555,Champions,...,0.049228,25890.416230,1274.540845,0.894717,Critical Risk,Monitor,Active Customers,1.000000,0.049228,0.762468
1,12415.0,98,15,102087.88,6805.86,5,5,5,555,Champions,...,0.061299,22728.702397,1393.253781,0.978052,Critical Risk,Monitor,Active Customers,0.877881,0.061299,0.759642
2,14646.0,92,42,178302.62,4245.30,5,5,5,555,Champions,...,0.048463,24336.187209,1179.401448,0.827930,Critical Risk,Monitor,Active Customers,0.939969,0.048463,0.711250
3,18102.0,98,27,135450.75,5016.69,5,5,5,555,Champions,...,0.050204,23071.894346,1158.303050,0.813119,Critical Risk,Monitor,Active Customers,0.891136,0.050204,0.687842
4,16684.0,132,18,33883.50,1882.42,4,5,5,455,Champions,...,0.098833,14413.396247,1424.518897,1.000000,Critical Risk,Monitor,VIP Customers,0.556708,0.098833,0.664614


In [3]:
# Keep only the columns that are useful for business decision-making
business_df = merged_df[[
    "CustomerID",
    "Segment",
    "Customer_Persona",
    "Churn",
    "Churn_Probability",
    "Predicted_CLV",
    "Revenue_at_Risk",
    "Risk_Score",
    "Risk_Category",
    "Customer_Action",
    "Priority_Score",
]].copy()

business_df["CustomerID"] = business_df["CustomerID"].astype(int)
for col in ["Predicted_CLV", "Revenue_at_Risk", "Priority_Score"]:
    business_df[col] = business_df[col].fillna(0)

business_df["Recommendation_Priority"] = pd.qcut(
    business_df["Priority_Score"], q=5, labels=["Low", "Medium", "High", "Very High", "Critical"], duplicates="drop"
)

business_df.head()

,CustomerID,Segment,Customer_Persona,Churn,Churn_Probability,Predicted_CLV,Revenue_at_Risk,Risk_Score,Risk_Category,Customer_Action,Priority_Score,Recommendation_Priority
0,17450,Champions,Active Customers,0,0.049228,25890.416230,1274.540845,0.894717,Critical Risk,Monitor,0.762468,Critical
1,12415,Champions,Active Customers,0,0.061299,22728.702397,1393.253781,0.978052,Critical Risk,Monitor,0.759642,Critical
2,14646,Champions,Active Customers,0,0.048463,24336.187209,1179.401448,0.827930,Critical Risk,Monitor,0.711250,Critical
3,18102,Champions,Active Customers,0,0.050204,23071.894346,1158.303050,0.813119,Critical Risk,Monitor,0.687842,Critical
4,16684,Champions,VIP Customers,0,0.098833,14413.396247,1424.518897,1.000000,Critical Risk,Monitor,0.664614,Critical


In [ ]:
# Groq setup
api_key = os.getenv("GROQ_API_KEY")
model_name = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

client = Groq(api_key=api_key) if api_key else None


def build_fallback_recommendation(row: Dict[str, Any]) -> str:
    risk_category = row.get("Risk_Category", "Medium")
    clv = float(row.get("Predicted_CLV", 0) or 0)
    churn = float(row.get("Churn_Probability", 0) or 0)

    if risk_category in {"High", "Very High", "Critical"} and clv > 5000:
        return "Prioritize a retention outreach plan for this high-value customer, combining premium service recovery with a personalized value proposition to protect revenue."
    if risk_category in {"High", "Very High", "Critical"}:
        return "Launch a targeted reactivation campaign with a compelling offer and clear customer success support to reduce churn risk."
    if churn > 0.6:
        return "Deploy a proactive engagement initiative focused on product value, support responsiveness, and near-term retention actions."
    return "Maintain a structured nurture sequence and closely monitor purchasing behavior for future upsell and loyalty opportunities."


def generate_with_groq(prompt: str, retries: int = 2) -> Optional[str]:
    if client is None:
        return None

    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an expert customer intelligence strategist for a revenue growth team. Write executive-ready recommendations that are concise, business-focused, and actionable for marketing, retention, and customer success leaders.",
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.2,
                max_tokens=400,
            )
            return response.choices[0].message.content.strip()
        except Exception as exc:
            logger.warning("Groq attempt %s failed: %s", attempt + 1, exc)
    return None


print(f"Groq configured: {'yes' if client else 'no'}")
print(f"Model: {model_name}")

Groq configured: yes
Model: llama-3.3-70b-versatile


In [ ]:
# Create recommendations for the highest-value customer opportunities
recommendation_rows = []
selected_customers = business_df.head(10).copy()

for _, row in selected_customers.iterrows():
    customer_id = int(row["CustomerID"])
    prompt = f"""
You are helping a revenue growth team prioritize customer retention actions.
Customer ID: {customer_id}
Segment: {row.get('Segment', 'Unknown')}
Persona: {row.get('Customer_Persona', 'Unknown')}
Churn Probability: {row.get('Churn_Probability', 0):.2%}
Predicted CLV: ${row.get('Predicted_CLV', 0):,.2f}
Revenue at Risk: ${row.get('Revenue_at_Risk', 0):,.2f}
Risk Category: {row.get('Risk_Category', 'Unknown')}
Customer Action: {row.get('Customer_Action', 'Monitor')}

Write a polished executive-style recommendation in one paragraph. Focus on the most appropriate action for retention, growth, or service recovery. Make it sound suitable for senior business stakeholders and include a clear rationale tied to the customer's risk and value.
"""

    llm_recommendation = generate_with_groq(prompt)
    recommendation_text = llm_recommendation or build_fallback_recommendation(row)

    recommendation_rows.append({
        "CustomerID": customer_id,
        "Segment": row.get("Segment", "Unknown"),
        "Customer_Persona": row.get("Customer_Persona", "Unknown"),
        "Churn_Probability": row.get("Churn_Probability", 0),
        "Predicted_CLV": row.get("Predicted_CLV", 0),
        "Revenue_at_Risk": row.get("Revenue_at_Risk", 0),
        "Risk_Score": row.get("Risk_Score", 0),
        "Risk_Category": row.get("Risk_Category", "Unknown"),
        "Customer_Action": row.get("Customer_Action", "Monitor"),
        "Priority_Score": row.get("Priority_Score", 0),
        "Recommendation_Priority": row.get("Recommendation_Priority", "Medium"),
        "Recommendation": recommendation_text,
    })

recommendations_df = pd.DataFrame(recommendation_rows).sort_values(["Priority_Score", "Predicted_CLV"], ascending=False).reset_index(drop=True)
recommendations_df.head(10)

2026-06-27 17:20:39,810 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-27 17:20:40,577 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-27 17:20:41,283 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-27 17:20:41,342 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-06-27 17:20:41,343 - WARNING - Groq attempt 1 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knc5wmfyehqanw6hd83rfenh` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99784, Requested 543. Please try again in 4m42.528s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
2026-06-27 17:20:41,393 - INFO - HTTP Request:

,CustomerID,Segment,Customer_Persona,Churn_Probability,Predicted_CLV,Revenue_at_Risk,Risk_Score,Risk_Category,Customer_Action,Priority_Score,Recommendation_Priority,Recommendation
0,17450,Champions,Active Customers,0.049228,25890.416230,1274.540845,0.894717,Critical Risk,Monitor,0.762468,Critical,"For Customer ID 17450, classified as a Champio..."
1,12415,Champions,Active Customers,0.061299,22728.702397,1393.253781,0.978052,Critical Risk,Monitor,0.759642,Critical,"For Customer ID 12415, a high-value Champion w..."
2,14646,Champions,Active Customers,0.048463,24336.187209,1179.401448,0.827930,Critical Risk,Monitor,0.711250,Critical,"For Customer ID 14646, a high-value Champion w..."
3,18102,Champions,Active Customers,0.050204,23071.894346,1158.303050,0.813119,Critical Risk,Monitor,0.687842,Critical,Maintain a standard nurture sequence and monit...
4,16684,Champions,VIP Customers,0.098833,14413.396247,1424.518897,1.000000,Critical Risk,Monitor,0.664614,Critical,Maintain a standard nurture sequence and monit...
5,14156,Champions,Active Customers,0.042543,23774.643653,1011.445265,0.710026,Critical Risk,Monitor,0.649418,Critical,Maintain a standard nurture sequence and monit...
6,14911,Champions,Active Customers,0.030983,22745.389773,704.721922,0.494709,Critical Risk,Monitor,0.536299,Critical,Maintain a standard nurture sequence and monit...
7,17511,Champions,Active Customers,0.039473,18841.964252,743.747483,0.522104,Critical Risk,Monitor,0.497557,Critical,Maintain a standard nurture sequence and monit...
8,12931,Champions,VIP Customers,0.100374,10139.915491,1017.780821,0.714473,Critical Risk,Monitor,0.478664,Critical,Maintain a standard nurture sequence and monit...
9,14088,Champions,VIP Customers,0.125216,7730.156361,967.938586,0.679485,Critical Risk,Monitor,0.435311,Critical,Maintain a standard nurture sequence and monit...


In [ ]:
# Executive summary generation
summary_prompt = f"""
You are summarizing the output of a customer intelligence program.
Create a concise executive summary for senior leadership that highlights the most important risks, opportunities, and recommended actions.

Business context:
- Top priority customers were selected from churn, CLV, persona, and revenue-at-risk signals.
- The highest-risk segment should be treated as an immediate retention opportunity.
- The recommendation table is saved to data/processed/customer_ai_recommendations.csv.

Key metrics:
- Customers reviewed: {len(recommendations_df)}
- Highest churn probability: {recommendations_df['Churn_Probability'].max():.2%}
- Highest predicted CLV: ${recommendations_df['Predicted_CLV'].max():,.2f}
- Highest revenue at risk: ${recommendations_df['Revenue_at_Risk'].max():,.2f}

Write a polished, executive-ready summary in 2 short paragraphs. Emphasize the biggest risks, the highest-value opportunities, and the recommended next actions for leadership. Keep the tone strategic and concise.
"""

executive_summary = generate_with_groq(summary_prompt)
if not executive_summary:
    executive_summary = (
        "The current customer portfolio shows a meaningful concentration of high-risk and high-value accounts that merit immediate action. "
        "The recommended workflow is to prioritize retention outreach for customers with elevated churn probability, high predicted CLV, and strong revenue-at-risk exposure."
    )

print(executive_summary)

2026-06-27 17:20:42,405 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-06-27 17:20:42,407 - WARNING - Groq attempt 1 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knc5wmfyehqanw6hd83rfenh` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99783, Requested 597. Please try again in 5m28.32s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
2026-06-27 17:20:42,455 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-06-27 17:20:42,456 - WARNING - Groq attempt 2 failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01knc5wmfyehqanw6hd83rfenh` service tier `on_demand` on tokens per day (TPD): Limit 100000, 

The current customer portfolio shows a meaningful concentration of high-risk and high-value accounts that merit immediate action. The recommended workflow is to prioritize retention outreach for customers with elevated churn probability, high predicted CLV, and strong revenue-at-risk exposure.


In [ ]:
# Save outputs
recommendations_df.to_csv(processed_dir / "customer_ai_recommendations.csv", index=False)

summary_path = processed_dir / "executive_summary.md"
summary_path.write_text(executive_summary, encoding="utf-8")

# Create a simple visualization for business review
plt.figure(figsize=(10, 5))
plt.barh(recommendations_df["CustomerID"].astype(str), recommendations_df["Priority_Score"], color="#2563eb")
plt.xlabel("Priority Score")
plt.ylabel("Customer ID")
plt.title("Top Customer Recommendation Priority")
plt.tight_layout()
plt.savefig(processed_dir / "recommendation_priority.png", dpi=200)
plt.close()

summary_payload = {
    "customers_reviewed": int(len(recommendations_df)),
    "highest_churn_probability": float(recommendations_df["Churn_Probability"].max()),
    "highest_predicted_clv": float(recommendations_df["Predicted_CLV"].max()),
    "highest_revenue_at_risk": float(recommendations_df["Revenue_at_Risk"].max()),
    "executive_summary": executive_summary,
}

with open(processed_dir / "ai_layer_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary_payload, handle, indent=2)

print("Saved outputs:")
print(f"- {processed_dir / 'customer_ai_recommendations.csv'}")
print(f"- {summary_path}")
print(f"- {processed_dir / 'recommendation_priority.png'}")
print(f"- {processed_dir / 'ai_layer_summary.json'}")

2026-06-27 17:20:42,562 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-06-27 17:20:42,563 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Saved outputs:
- c:\Users\vkt58\OneDrive\Desktop\Customer-Intelligence-Revenue-Growth\data\processed\customer_ai_recommendations.csv
- c:\Users\vkt58\OneDrive\Desktop\Customer-Intelligence-Revenue-Growth\data\processed\executive_summary.md
- c:\Users\vkt58\OneDrive\Desktop\Customer-Intelligence-Revenue-Growth\data\processed\recommendation_priority.png
- c:\Users\vkt58\OneDrive\Desktop\Customer-Intelligence-Revenue-Growth\data\processed\ai_layer_summary.json


: 

## Business Insights

The generated recommendation table highlights the customers most likely to benefit from retention and growth actions. In practice, these accounts should be prioritized for outreach, premium retention offers, or proactive service interventions.

## Business Conclusion

When your Groq key is available, this notebook will generate richer, more tailored recommendations from the LLM. Until then, the deterministic fallback provides a robust baseline and produces the same artifacts for reporting and operations.